
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>





# Propagating Changes with CDF Lab

We'll be using Change Data Feed to propagate changes to many tables from a single source.

For this lab, we'll work with the fitness tracker datasets to propagate changes through a Lakehouse with Delta Lake Change Data Feed (CDF).

Because the **`user_lookup`** table links identifying information between different pipelines, we'll make this the point where changes propagate from.


## Objectives
By the end of this lab, you should be able to:
- Enable Change Data Feed on a particular table
- Read CDF output with Spark SQL or PySpark
- Refactor ELT code to process CDF output




Begin by running the following cell to set up relevant databases and paths.

In [0]:
%run ./Includes/Classroom-Setup-03.5L


## Enables CDF for the table

To enables CDF for the **"user_lookup"** table use ALTER TABLE and set TBLPROPERTIES to activate **`delta.enableChangeDataFeed`**.

In [0]:
%sql
ALTER TABLE user_lookup 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);



## Read the CDF output from table

To read the CDF data:
- Set up a streaming read on the **`user_lookup`** table
- Configure the stream to enable reading change data
- Configure the stream to start reading from version 1 of the **`user_lookup`** table

In [0]:
user_lookup_df = (spark.readStream
           .format("delta")
           .option("readChangeData", True)
           .option("startingVersion", 1)
           .table("user_lookup"))

display(user_lookup_df)

### Delete a record from table

To delete record from table:
- Use **`DELETE`** statement with column_name of table
- Enter actual name of the column and value to delete record

In [0]:
%sql
DELETE FROM user_lookup WHERE user_id = 49661

### Check that the record was deleted from user_lookup

To check whether record was deleted:
- Use **`SELECT`** statement to get record from table
- Specify the column_name and value to see whether record with specific id exist in table

In [0]:
%sql
SELECT * FROM user_lookup WHERE user_id = 49661

### Propagate deletes from multiple tables

To propagate delete follow these steps:
- Create temporary view of user_lookup table as **`user_lookup_deletes`**
- Select all record in view where **`_change_type`** is **delete**  
- Merge into **`users`** table when **`alt_id`** gets matched
- Similarly, merge into **`user_bins`** table when **`user_id`** gets matched

In [0]:
%sql

-- Create a temporary view for change data with delete entries

CREATE OR REPLACE TEMPORARY VIEW user_lookup_deletes AS
SELECT *
FROM table_changes("user_lookup", 1)
WHERE _change_type = 'delete';

-- Apply deletions to the "user" and "user_bin" tables
MERGE INTO users u
USING user_lookup_deletes uld
ON u.alt_id = uld.alt_id
WHEN MATCHED
 THEN DELETE;

MERGE INTO user_bins ub
USING user_lookup_deletes uld
ON ub.user_id = uld.user_id
WHEN MATCHED
 THEN DELETE;


In [0]:
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>